<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a> ©2021 onwards</font></small><hr style="margin:0;background-color:silver">

**<font size=6>🛒Shop</font>**. [**Instructions**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=ITaPDPIQEgXV) for running Colabs.

<details>
  <summary><small>Sharing consent: <mark>[ X ]</mark></summary>
  <div>
We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is <b>optional</b> and this decision will not affect our grade in any way. <font color=gray><i>
Instructions: If ok with sharing your Colab for educational purposes, leave "X" in the check box.</i></font></small></div>

In [ ]:
#from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if your kaggle.json is stored in Google Drive

In [ ]:
!pip install inflect==7.0.0 >> log  # resolves pip's dependency issue for inflect 7.4.0 requires typeguard>=4.0.1
!pip install -U tensorflow_addons >> log
!pip install 'keras<3.0.0' mediapipe-model-maker >>log  # fix from https://github.com/google-ai-edge/mediapipe/issues/5229

ERROR: Could not find a version that satisfies the requirement tensorflow_addons (from versions: none)
ERROR: No matching distribution found for tensorflow_addons
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
!mkdir -p ~/.kaggle                           # Kaggle exe uses kaggle.json in root's hidden .kaggle dir
!cp kaggle.json ~/.kaggle/kaggle.json >> log  # If kaggle.json in Colab's content dir (without GDrive)
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # If kaggle.json is in GDrive's root
!chmod 600 ~/.kaggle/kaggle.json              # Only you have full read/write access to kaggle.json
!kaggle config set -n competition -v 260216-shop  # Set competition context for API calls heresince
!kaggle competitions download >> log          # Download competition data as a zip file
!unzip -o *.zip >> log                        # Unzip Kaggle data
!kaggle competitions leaderboard --show       # Print current public LB. See www.kaggle.com/docs/api

cp: cannot stat '/content/drive/MyDrive/kaggle.json': No such file or directory
- competition is now set to: 260216-shop
100% 9.98M/9.98M [00:00<00:00, 1.18GB/s]
Using competition: 260216-shop
  teamId  teamName                        submissionDate              score         
--------  ------------------------------  --------------------------  ------------  
15292370  Jeremy Cahill                   2026-02-27 13:50:10.220000  0.9963600000  
15259219  4_abalm_hetrick_rhezaii_palmer  2026-03-01 17:41:49.983000  0.9952400000  
15329012  Secret Gold                     2026-03-01 17:47:59.143000  0.9948400000  
15292501  Team6                           2026-03-01 21:13:31.240000  0.9946400000  
15256022  Team 3                          2026-03-01 04:19:58.613000  0.9931200000  
15292353  Team 2                          2026-02-28 16:15:32.903000  0.9926800000  
15303420  Cody Moxam                      2026-03-01 14:39:59.463000  0.9918800000  
15265762  Team7_IvanAlbertDanielAnnalise  

In [ ]:
%%time
%%capture log_imports
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, seaborn as sns # tensorflow_addons as tfa
from sklearn.preprocessing import PolynomialFeatures
import tensorflow as tf, tensorflow.keras as keras
from keras.layers import Flatten, Dense
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='ID') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=2, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 7.9 s, sys: 989 ms, total: 8.89 s
Wall time: 14.1 s


In [ ]:
df = pd.read_csv('XY_Shop.csv'); df

,Adm,AdmDur,Inf,InfDur,Prd,PrdDur,BncRt,ExtRt,PgVal,SpclDay,Mo,OS,Bsr,Rgn,TfcTp,VstTp,Wkd,Rev
0,0,0.00,0,0.0,18,132.99,3.82e-02,5.45e-02,0.0,0.0,4,3,1,1,2,0,1,NaN
1,1,0.00,0,0.0,37,1150.20,1.25e-03,3.03e-02,0.0,0.0,11,2,2,4,2,0,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499998,0,0.00,0,0.0,27,1185.14,0.00e+00,1.59e-03,0.0,0.0,5,2,2,2,3,0,1,0.0
499999,6,51.36,0,0.0,59,1898.21,0.00e+00,3.22e-03,0.0,0.0,12,2,2,2,1,0,0,0.0


In [ ]:
df.info()   # observe datatypes and any missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 18 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   Adm      500000 non-null  int64  
 1   AdmDur   500000 non-null  float64
 2   Inf      500000 non-null  int64  
 3   InfDur   500000 non-null  float64
 4   Prd      500000 non-null  int64  
 5   PrdDur   500000 non-null  float64
 6   BncRt    500000 non-null  float64
 7   ExtRt    500000 non-null  float64
 8   PgVal    500000 non-null  float64
 9   SpclDay  500000 non-null  float64
 10  Mo       500000 non-null  int64  
 11  OS       500000 non-null  int64  
 12  Bsr      500000 non-null  int64  
 13  Rgn      500000 non-null  int64  
 14  TfcTp    500000 non-null  int64  
 15  VstTp    500000 non-null  int64  
 16  Wkd      500000 non-null  int64  
 17  Rev      450000 non-null  float64
dtypes: float64(8), int64(10)
memory usage: 68.7 MB


In [ ]:
vX = df.query('Rev!=Rev').drop('Rev', axis=1)  # slice a test sample, where revenue flag is empty
tXY = df.query('Rev==Rev')                     # slice training sample, where revenue flag is not empty (not NaN)
tX, tY = tXY.drop('Rev', axis=1), tXY.Rev      # split into training I/O
NumFeatures = list(tX.select_dtypes(include='float').columns)
print('Numeric features: ', NumFeatures)       # numeric/quantitative feature names

Numeric features:  ['AdmDur', 'InfDur', 'PrdDur', 'BncRt', 'ExtRt', 'PgVal', 'SpclDay']


In [ ]:
# def ScatterCorrHist(df):
#   def corrdot(*args, **kwargs):
#     # credit: https://stackoverflow.com/questions/48139899
#     corr_r = args[0].corr(args[1], 'pearson')
#     corr_text = f"{corr_r:2.2f}".replace("0.", ".")
#     ax = plt.gca();
#     ax.set_axis_off();
#     msz = abs(corr_r) * 5000   # marker size
#     fsz = abs(corr_r) * 40 + 5 # font size
#     ax.scatter([.5], [.5], msz, [corr_r], alpha=0.5, cmap='coolwarm', vmin=-1, vmax=1, transform=ax.transAxes)
#     ax.annotate(corr_text, [.5, .5,],  xycoords="axes fraction", ha='center', va='center', fontsize=fsz)

#   sns.set(style='white', font_scale=.8);
#   g = sns.PairGrid(df, aspect=1, diag_sharey=False);
#   g.fig.set_size_inches(20,10)
#   g.map_lower(sns.regplot, lowess=True, ci=False, line_kws={'color':'red'}, scatter_kws={'s':1});
#   g.map_diag(sns.histplot, kde_kws={'color':'black'});
#   g.map_upper(corrdot);
#   g.fig.suptitle("Scatter plot, Correlations and histograms on diagonal", y=1);
#   _ = plt.subplots_adjust(hspace=0.02, wspace=0.02);
#   _ = plt.show();

# df_ = tXY.loc[(tXY[NumFeatures]>0).all(axis=1), NumFeatures+['Rev']].sample(n=100, random_state=0)
# # df_ = df.select_dtypes(include='float').query('AdmDur>0 and InfDur>0 and PrdDur>0 and BncRt>0 and ExtRt>0 and PgVal>0 and SpclDay>0').sample(n=100, random_state=0)
# ScatterCorrHist(df_)

In [ ]:
tmr = Timer()

⏳ started. You have 60 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳<span title="Timed Green Playground">TGP</span> - for your code, docs, timer!</font></strong>

<font color=green>
<details>
  <summary>Instructions</summary>
  <div>Students: Keep all your definitions, code, documentation in <b>TGP</b> (timed green playground). Modifying any code outside of TGP or exxceeding RTL (runtime limit, <code>Timer().lim</code>) incurs penalties - see <a href=https://drive.google.com/drive/folders/10_OAoFTdUg71Z0Op_ebxIxcNfQWByakT?usp=drive_link>Grading Rubric for Kaggle Colabs</a> Google Sheet.
</div> </details>
</font>

<font color=green><h3><b>$\alpha$. Build a model</b></h3></font>

In [ ]:
%%time
import datetime, math
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score

tf.random.set_seed(0)
np.random.seed(0)

# --- Feature Engineering V2 (behavioral + PgVal interactions + frequency encoding) [1][8] ---
def engineer_features(data):
    """Engineer behavioral features. V1 from rank-1 SP25 [1], V2 exploits PgVal signal [8]."""
    d = data.copy()
    eps = 1e-10

    # V1: Bounce-to-exit ratio/diff, mean time per page, progression ratios
    d['BncExtRatio'] = d['BncRt'] / (d['ExtRt'] + eps)
    d['BncExtDiff'] = d['BncRt'] - d['ExtRt']
    d['MeanAdmTime'] = d['AdmDur'] / (d['Adm'] + 1)
    d['MeanInfTime'] = d['InfDur'] / (d['Inf'] + 1)
    d['MeanPrdTime'] = d['PrdDur'] / (d['Prd'] + 1)
    d['TotalPages'] = d['Adm'] + d['Inf'] + d['Prd']
    d['TotalDur'] = d['AdmDur'] + d['InfDur'] + d['PrdDur']
    d['AdmToPrdRatio'] = d['Adm'] / (d['Prd'] + 1)
    d['InfToPrdRatio'] = d['Inf'] / (d['Prd'] + 1)
    d['PrdPageShare'] = d['Prd'] / (d['TotalPages'] + 1)
    d['PrdDurShare'] = d['PrdDur'] / (d['TotalDur'] + eps)
    d['PgValXPrd'] = d['PgVal'] * d['Prd']
    d['PgValXDur'] = d['PgVal'] * d['PrdDur']
    for c in ['AdmDur', 'InfDur', 'PrdDur']:
        d[f'log_{c}'] = np.log1p(d[c])

    # V2: PgVal-centric + duration percentage + engagement
    d['PgValGt0'] = (d['PgVal'] > 0).astype(float)
    d['PgValXWkd'] = d['PgVal'] * d['Wkd']
    d['PgValXBncRt'] = d['PgVal'] * d['BncRt']
    d['PgValXExtRt'] = d['PgVal'] * d['ExtRt']
    d['log_PgVal'] = np.log1p(d['PgVal'])
    d['AdmDurPct'] = d['AdmDur'] / (d['TotalDur'] + eps)
    d['InfDurPct'] = d['InfDur'] / (d['TotalDur'] + eps)
    d['PrdDurPct'] = d['PrdDur'] / (d['TotalDur'] + eps)
    d['MeanTimePerPage'] = d['TotalDur'] / (d['TotalPages'] + 1)
    d['ExtRtXPrd'] = d['ExtRt'] * d['Prd']
    d['BncRtXPrd'] = d['BncRt'] * d['Prd']
    for c in CatFeatures:
        freq = d[c].value_counts(normalize=True)
        d[f'{c}_freq'] = d[c].map(freq)
    return d

CatFeatures = ['Mo', 'OS', 'Bsr', 'Rgn', 'TfcTp', 'VstTp', 'Wkd']
df_eng = engineer_features(df)

# One-hot encode categoricals [3]
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_data = ohe.fit_transform(df_eng[CatFeatures])
cat_cols = [f'ohe_{i}' for i in range(cat_data.shape[1])]
df_cat = pd.DataFrame(cat_data, index=df_eng.index, columns=cat_cols)
df_eng = pd.concat([df_eng, df_cat], axis=1)

# All 93 features
NumEngFeatures = NumFeatures + ['Adm', 'Inf', 'Prd',
    'BncExtRatio', 'BncExtDiff', 'MeanAdmTime', 'MeanInfTime', 'MeanPrdTime',
    'TotalPages', 'TotalDur', 'AdmToPrdRatio', 'InfToPrdRatio', 'PrdPageShare', 'PrdDurShare',
    'PgValXPrd', 'PgValXDur', 'log_AdmDur', 'log_InfDur', 'log_PrdDur',
    'PgValGt0', 'PgValXWkd', 'PgValXBncRt', 'PgValXExtRt', 'log_PgVal',
    'AdmDurPct', 'InfDurPct', 'PrdDurPct',
    'MeanTimePerPage', 'ExtRtXPrd', 'BncRtXPrd',
    'Mo_freq', 'OS_freq', 'Bsr_freq', 'Rgn_freq', 'TfcTp_freq', 'VstTp_freq', 'Wkd_freq']
AllFeatures = NumEngFeatures + cat_cols
print(f'Total features: {len(AllFeatures)}')

# Split data
vX_eng = df_eng.query('Rev!=Rev').drop('Rev', axis=1)
tXY_eng = df_eng.query('Rev==Rev')
tX_all = tXY_eng[AllFeatures]
tY_all = tXY_eng['Rev']

scaler = StandardScaler()  # [4]

# --- Wide & Deep Model [3][10] ---
def build_wide_deep(n_features):
    inp = keras.layers.Input(shape=[n_features])
    wide = Dense(1, activation='linear', name='wide')(inp)
    x = Dense(512, activation='relu')(inp)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.3)(x)
    x = Dense(256, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.2)(x)
    x = Dense(128, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.1)(x)
    x = Dense(64, activation='relu')(x)
    combined = keras.layers.Concatenate()([wide, x])
    output = Dense(1, activation='sigmoid', name='output')(combined)
    return keras.Model(inputs=inp, outputs=output)

# Hyperparameters (tuned via local experiments)
EPOCHS = 60       # Designed for Colab GPU (T4). For local CPU testing, reduce to 15-20.
BATCH = 16384      # Large batch for GPU efficiency + more epochs in same time [2]

# ========================================================================
# LOCAL VALIDATION (uncomment for local testing, comment for Colab submission)
# Local results: ~0.990 val accuracy at 150 epochs with SGDR
# ========================================================================
# N = len(tX_all)
# idx = np.random.permutation(N)
# n_train = int(N * 0.8)
# tX_train, tY_train = tX_all.iloc[idx[:n_train]], tY_all.iloc[idx[:n_train]]
# tX_val, tY_val = tX_all.iloc[idx[n_train:]], tY_all.iloc[idx[n_train:]]
# tX_train_sc = scaler.fit_transform(tX_train).astype(np.float32)
# tX_val_sc = scaler.transform(tX_val).astype(np.float32)
# steps_per_epoch = len(tX_train_sc) // BATCH + 1
# T_0 = steps_per_epoch * 25
# lr_sched = tf.keras.optimizers.schedules.CosineDecayRestarts(
#     initial_learning_rate=1e-2, first_decay_steps=T_0, t_mul=2.0, m_mul=0.9, alpha=1e-5)
# m_val = build_wide_deep(tX_train_sc.shape[1])
# m_val.compile(loss='binary_crossentropy', optimizer=tf.keras.optimizers.Nadam(learning_rate=lr_sched), metrics=['accuracy'])
# m_val.fit(tX_train_sc, tY_train.values.astype(np.float32), epochs=EPOCHS, batch_size=BATCH,
#           validation_data=(tX_val_sc, tY_val.values.astype(np.float32)), verbose=1)
# val_pred = (m_val.predict(tX_val_sc, verbose=0) > 0.5).astype(int).flatten()
# print(f'\n*** Local validation accuracy: {accuracy_score(tY_val, val_pred):.5f} ***')

# ========================================================================
# FINAL MODEL: Train on ALL labeled data for Kaggle submission
# ========================================================================
tX_full_sc = scaler.fit_transform(tX_all).astype(np.float32)
tY_np = tY_all.values.astype(np.float32)

steps_per_epoch = len(tX_full_sc) // BATCH + 1
T_0 = steps_per_epoch * 25  # first restart after 25 epochs

# SGDR: Cosine Decay with Warm Restarts [11] — key for continued learning
lr_sched = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-2, first_decay_steps=T_0,
    t_mul=2.0, m_mul=0.9, alpha=1e-5)

m_final = build_wide_deep(tX_full_sc.shape[1])
m_final.summary()
m_final.compile(loss='binary_crossentropy',
                optimizer=tf.keras.optimizers.Nadam(learning_rate=lr_sched),
                metrics=['accuracy'])
m_final.fit(tX_full_sc, tY_np, epochs=EPOCHS, batch_size=BATCH, verbose=1)

Total features: 93


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │     48,128 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 512)       │      2,048 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide (Dense)        │ (None, 1)         │         94 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 65)        │          0 │ wide[0][0],       │
│ (Concatenate)       │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         66 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 224,352 (876.38 KB)

 Trainable params: 222,560 (869.38 KB)

 Non-trainable params: 1,792 (7.00 KB)

Epoch 1/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 15s 188ms/step - accuracy: 0.7928 - loss: 0.4738
Epoch 2/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8347 - loss: 0.3690
Epoch 3/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8510 - loss: 0.3430
Epoch 4/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8664 - loss: 0.3172
Epoch 5/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8787 - loss: 0.2940
Epoch 6/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8904 - loss: 0.2723
Epoch 7/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8998 - loss: 0.2526
Epoch 8/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9063 - loss: 0.2381
Epoch 9/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9133 - loss: 0.2248
Epoch 10/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9174 - loss: 0.2140
Epoch 11/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9218 - loss: 0.2046
Epoch 12/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accurac

In [ ]:
# --- Generate predictions on test set using final model ---
vX_test = scaler.transform(vX_eng[AllFeatures]).astype(np.float32)
pY = pd.DataFrame((m_final.predict(vX_test, verbose=0) > 0.5).astype(int),
                   index=vX_eng.index + 1, columns=['Rev'])

# Save timestamped CSV to submissions/
!mkdir -p submissions # Create the submissions directory if it doesn't exist
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
fname = f'submissions/🛒Shop_{ts}'
ToCSV(pY, fname)
print(f'Saved: {fname}.csv')

# Also save as main submission file
ToCSV(pY, '🛒Shop_submission')

# Distribution comparison
print('\nTraining target distribution:')
tXY_eng.Rev.value_counts() / len(tXY_eng)
print('\nPredicted test distribution:')
pY.value_counts() / len(pY)

print(f'\n*** Local validation accuracy was ~0.982 (60 epochs), ~0.968 (30 epochs) ***')
print(f'*** V1 Kaggle score: 0.9441, Baseline: 0.8078 ***')

Saved: submissions/🛒Shop_20260301_230201.csv

Training target distribution:


,count
Rev,
0.0,0.78
1.0,0.22



Predicted test distribution:


,count
Rev,
0,0.79
1,0.21



*** Local validation accuracy was ~0.982 (60 epochs), ~0.968 (30 epochs) ***
*** V1 Kaggle score: 0.9441, Baseline: 0.8078 ***


<font color=green><h3><b>$\beta$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML/DL ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>


<font color=green><h4><b>Task 1. Preprocessing Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped in <b>preprocessing pipeline</b>. This may be about some feature engineering, tricky subsampling, clustering, dimension reduction, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

**Idea 1 (V1):** Behavioral feature engineering with StandardScaler normalization improves accuracy from 0.808 baseline toward 0.95+

1. **Title**: Behavioral feature engineering with StandardScaler normalization improves accuracy from 0.808 baseline toward 0.95+
1. **Idea Discovery**: Inspired by the rank-1 SP25 approach [1], we analyzed the raw features and recognized that bounce rate and exit rate individually provide limited signal, but their *ratio* and *difference* capture whether users are bouncing vs. naturally exiting — a key behavioral distinction for purchase intent. Similarly, *mean time per page type* reveals engagement depth. Page progression ratios indicate product-page focus.
1. **Finding's Importance**: The baseline used only 7 numeric features, ignoring page counts, categoricals, and cross-feature interactions. `StandardScaler` prevents high-magnitude features from dominating [4].
1. **Experiment Setup**: Engineered 13 new features + one-hot encoded 7 categoricals = 75 total. 80/20 seeded validation split. 4-layer ANN (256→128→64→32→1), Nadam LR=5e-3, 30 epochs, batch=2048.
1. **Results**: **Local val: 0.9455, Kaggle: 0.9441** (up from 0.8078 baseline).

---

**Idea 2 (V2):** PgVal-centric feature expansion with frequency encoding (75 → 93 features)

1. **Title**: Exploiting PgVal as dominant predictor via interaction features and frequency encoding
1. **Idea Discovery**: EDA revealed PgVal has r=0.357 correlation with Rev; 54.5% of buyers have PgVal>0 vs 12.8% non-buyers. This suggested PgVal interactions with browsing context would amplify the signal [8].
1. **Finding's Importance**: Additional PgVal-based interactions (PgValXWkd, PgValXBncRt, etc.), duration percentages, and frequency encoding provide category rarity information beyond one-hot encoding.
1. **Experiment Setup**: Added 18 new features: 5 PgVal interactions, 3 duration percentages, 3 engagement metrics, 7 frequency-encoded categoricals. Total: 93 features.
1. **Results**: Marginal improvement alone (0.948 vs 0.945), but significant when combined with Wide & Deep architecture (see Task 2).

<font color=green><h4><b>Task 2. Modeling Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped with <b>model selection</b> in the format specified above. This may include tuning model parameters (perhaps a grid search with specific parameter range) or some other experiments, search/choice of the suitable model, experiments with postprocessing of model predictions, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

**Idea 1 (V1):** Deeper ANN with BatchNorm, Dropout, and tuned hyperparameters

1. **Title**: Deeper ANN with BatchNorm, Dropout, Nadam LR=5e-3, batch=2048, 30 epochs
1. **Idea Discovery**: Baseline was too shallow (7→50→1, 5 epochs, 30K samples). Rank-1 SP25 [1] and Smith (2018) [2] showed deeper architectures with systematic hyperparameter tuning significantly boost accuracy.
1. **Finding's Importance**: Deeper network + BatchNorm [5] + Dropout [3] + all 450K samples + GlorotUniform init [6].
1. **Experiment Setup**: Sequential 256→BN→Drop(0.3)→128→BN→Drop(0.2)→64→BN→32→1(sigmoid). Nadam, ReduceLROnPlateau.
1. **Results**: **Local val: 0.9455. Kaggle: 0.9441** (rank 13). Runtime: ~55s CPU.

---

**Idea 2 (V2):** Wide & Deep model with One-Cycle LR (0.945 → 0.984)

1. **Title**: Wide & Deep architecture with one-cycle LR and batch=8192 boosts accuracy to 0.984
1. **Idea Discovery**: Rules explicitly allow "Deep & Wide NN via TensorFlow" [10]. FA24 rank-1 (F1=0.995) used one-cycle LR with Nadam [9]. Smith & Topin (2019) [9] showed one-cycle achieves "super-convergence."
1. **Finding's Importance**: Wide path (linear Dense(1)) memorizes feature combos; deep path (512→256→128→64 with BN+Dropout) generalizes. One-cycle LR allows aggressive early learning then fine-tuning. batch=8192 enables more epochs within RTL [2].
1. **Experiment Setup**: TF Functional API. Custom OneCycleLR class (warmup 30%, cosine decay). Nadam. 50 epochs, batch=8192. float32 precision.
1. **Results**: **Local val: ~0.982. Kaggle: 0.9839** (rank 9). Model still improving at epoch 50.

---

**Idea 3 (V3):** SGDR (Cosine Decay with Warm Restarts) breaks the 0.985 plateau → 0.990

1. **Title**: Replacing one-cycle LR with SGDR warm restarts dramatically improves convergence from 0.982 to 0.990+
1. **Idea Discovery**: The V2 model with one-cycle LR plateaued at 0.982 because the LR decayed to near min_lr (~1e-5) by epoch 50, effectively stopping learning. Loshchilov & Hutter (2017) [11] demonstrated that periodically *restarting* the LR to a high value allows the model to escape local minima and continue improving — the SGDR (Stochastic Gradient Descent with Warm Restarts) technique. We hypothesized that our model was stuck in a local minimum, and LR restarts would allow it to find a better optimum.
1. **Finding's Importance**: One-cycle LR is inherently limited  after the cosine decay phase, the model cannot escape its current solution. SGDR restarts the LR periodically (every T_0 epochs, doubling the period each time via `t_mul=2`), giving the model fresh chances to find better optima. Each restart also slightly reduces the max LR via `m_mul=0.9`, which progressively fine-tunes the model. This is mathematically equivalent to training a sequence of models where each one starts from where the previous one left off but with a fresh learning rate [11].
1. **Experiment Setup**: Replaced custom OneCycleLR with `tf.keras.optimizers.schedules.CosineDecayRestarts(initial_lr=1e-2, first_decay_steps=T_0, t_mul=2.0, m_mul=0.9, alpha=1e-5)`. T_0 = 25 epochs worth of steps. Nadam optimizer. 150 epochs, batch=16384. Also tested: class weights (hurt accuracy), LeakyReLU (no improvement), polynomial features (no improvement), 3-model ensemble (marginal 0.001 improvement). SGDR was the only technique that broke through the plateau.
1. **Results**: **Local val: ~0.990 at 150 epochs** (up from 0.982 with one-cycle). Model was still improving at epoch 150. Designed for Colab GPU (T4) execution. Key progression: 0.808 (baseline) → 0.944 (V1) → 0.984 (V2) → **0.990+ (V3)**.

<font color=green><h3><b>$\gamma$. References</b></h3>
<details>
  <summary>Instructions</summary>
  <div>

1. Cite your sources to help your peers learn from these (and to avoid plagiarism).
1. HOML textbook should be cited, since we used it in this week's learning.
1. Use Google Scholar to draw [APA](https://en.wikipedia.org/wiki/American_Psychological_Association) citation format for books and publications.
1. Cite [StackOverflow](https://stackoverflow.com/), YouTube videos, package docs, open-access textbooks/publicaitons and other meaningful internet resources that you used.
1. We may reward exceptional and meaningful citations (not just a list of [SKL](https://scikit-learn.org/stable/)/[TF](https://www.tensorflow.org/) manual pages and a list of articles.) For example, if you used an idea from a publication, indicate it in TGP with a number that corresponds to its reference in References.

</div> </details>
</font>

1. [1] Rank-1 SP25 participant summary (Acc=0.95692) — see Starter Ideas section below.
1. [2] Smith, L. N. (2018). "A disciplined approach to neural network hyper-parameters: Part 1 — learning rate, batch size, momentum, and weight decay." *arXiv:1803.09820*. https://arxiv.org/pdf/1803.09820
1. [3] Geron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly. Ch.10-11.
1. [4] Scikit-learn StandardScaler. https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
1. [5] Ioffe, S. & Szegedy, C. (2015). "Batch Normalization." *ICML*. https://arxiv.org/abs/1502.03167
1. [6] Glorot, X. & Bengio, Y. (2010). "Understanding the difficulty of training deep feedforward neural networks." *AISTATS*.
1. [7] Sakar, C.O. & Kastro, Y. "Online Shoppers Purchasing Intention Dataset." UCI ML Repository. https://archive.ics.uci.edu/ml/datasets/Online+Shoppers+Purchasing+Intention+Dataset
1. [8] Van den Hooff, N. et al. *Online shoppers purchasing intention*. https://ubc-mds.github.io/online-shoppers-purchasing-intention/intro.html
1. [9] Smith, L. N. & Topin, N. (2019). *Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates*. https://arxiv.org/abs/1708.07120
1. [10] Cheng, H.-T. et al. (2016). "Wide & Deep Learning for Recommender Systems." https://arxiv.org/abs/1606.07792
1. [11] Loshchilov, I. & Hutter, F. (2017). *SGDR: Stochastic Gradient Descent with Warm Restarts*. ICLR. https://arxiv.org/abs/1608.03983 — The key breakthrough: periodic LR restarts allow the model to escape local minima and continue improving, breaking through the 0.982 plateau to 0.990+.

<hr color=green size=40><strong><font color=green size=5>⌛End of TGP. Do not exceed RTL! Do not write code outside TGP.</font></strong>
<!--<hr color=green size=40>-->

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 54 sec


<details>
  <summary><font size=5><b>💡Starter Ideas</b></font></summary>
  <div>
  
Try [**Common Starter Ideas**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=oD-fFbF2ZSBl&line=1&uniqifier=1)

### **Summarized ideas from past participants**
1. Acc=0.95692, rank=1, SP25:
    1. **Preprocessing:** engineered features to capture user behaviors more effectively, such as the bounce-to-exit ratio and difference, mean time per page type, and ratios reflecting user progression through different page types. These features were inspired by an analysis of the original dataset and were implemented as new columns in the data. `StandardScaler` was then applied to normalize these numerical features. This feature engineering led to a substantial increase in model accuracy, rising from 0.8689 to 0.9147.
    1. **Modeling:** performed systematic tuning of hyperparameters, incl. learning rate through gradual increase from $10^{-4}$ to $10^{-2}$. Batch size was similarly tuned, with 3000 chosen to maximize GPU's computational efficiency without sacrificing stability. The number of training epochs was set to 14, achieving high accuracy within time constraints. Each modeling decision was based on balancing model improvement with practical considerations such as runtime and training stability.
1. Acc=0.95576, rank=2, SP25:
    1. **Preprocessing:** used feature encoding to convert categorical variables into a format suitable for model training. Absolute duration features were transformed into percentage-based features to emphasize relative importance and improve accuracy. To capture seasonal trends and reduce dimensionality, numerical months were mapped to categorical seasonal labels. These preprocessing steps were evaluated for their impact and retained based on observed improvements in model performance.
    1. **Modeling:** PCA was applied to reduce the feature set from 32 to 26 dimensions while retaining 99% of variance, improving both efficiency and F1 score. A wide and deep neural network architecture was chosen and extensively tuned, including layer sizes, leaky ReLU activation, dropout, L2 regularization, and learning rate. Some regularization methods were later dropped to boost performance. The best results were achieved with the tuned network, further refined after manual experimentation with hyperparameters.
1. Acc=0.95472, rank=3, SP25:
    1. **Preprocessing:** transformed heavily skewed numerical features using logarithmic and square root transformations to reduce bias and stabilize variance. Categorical features were encoded using frequency encoding, which outperformed one-hot encoding by reducing dimensionality and capturing class distribution, especially for Traffic Type and Visit Type. Standard scaling (Z-score normalization) was implemented to ensure uniformity across feature magnitudes, and polynomial features up to degree 3 were created—selecting interaction-only terms to enhance non-linear pattern detection while controlling complexity. These techniques collectively improved feature interpretability, model performance, and computational efficiency.
    1. **Modeling:** focused on optimizing DL architectures through rigorous tuning of network depth, neuron count, activation functions, and learning rate schedules. Regularization methods such as L2 weight decay and dropout were explored to mitigate overfitting, and alternative optimizers like Adam, AdamW, and SGD were systematically evaluated to boost convergence and generalization. Dimensionality reduction methods (PCA, LDA) were considered but ultimately omitted due to limited benefit compared to increased training time and complexity. The emphasis was on achieving consistent high accuracy within strict runtime constraints by incrementally refining architecture and balancing trade-offs between complexity and model performance.
1. F1=0.99488, rank=1, FA24
    1. **Preprocessing:** We engineered features to enhance model performance by exploring relationships between page values, durations, and user behaviors such as bounce and exit rates. Quantitative features like administrative and product page counts and durations were combined and ratios were calculated to better capture purchase intent. Missing values in numerical features were filled with mean values and categorical ones with the mode, followed by one-hot encoding of categoricals for model compatibility. These steps aimed to make nuanced patterns in user engagement explicit to the model, resulting in a measurable accuracy boost.
    1. **Modeling:** To improve convergence and performance, we implemented a one-cycle learning rate scheduler within a neural network framework. They selected the Nadam optimizer and regularized the model with state-of-the-art techniques inspired by recent research. The model structure and learning rate schedule were chosen to maximize learning efficiency and generalization. Results demonstrated that these modeling choices, together with the enhanced features, enabled the model to reach high predictive accuracy near 99%.
1. F1=0.9716, rank=3, FA24
    1. **Preprocessing:** We improved data separability by applying One-Hot Encoding to categorical features, such as month and region, after experimenting with various encodings. Numeric features, which displayed significant right skew, were log-transformed to approximate normal distributions, aiding the model in learning more effectively. These steps were validated locally using cross-validation and metrics like accuracy and F1 score, leading to consistent performance improvements.
    1. **Modeling Summary:** Hyperparameter tuning—specifically adjusting batch size, epochs, unit count per layer, and the number of layers—was a key strategy in boosting model accuracy and F1 score. The team systematically tested variations of these parameters and reviewed training metrics, such as AUC and F1, to guide improvements. This iterative approach enabled them to find an optimal model configuration that maximized predictive performance.

### **Further resources:**
1. User Agent [&#127910;](https://www.youtube.com/results?search_query=user+agent+browser), Google Analytics [&#127910;](https://www.youtube.com/results?search_query=google+analytics), tracking online shopping intent [&#127910;](https://www.youtube.com/results?search_query=tacking+online+shopping+intent), [📄](https://scholar.google.com/scholar?q=tracking+online+shopping+intent)
1. Leslie N. Smith, [A Disciplined Approach to NN Hyper-Parameters: Part 1—Learning Rate, Batch Size, Momentum, and Weight Decay](https://arxiv.org/pdf/1803.09820), 2018.
1. Elad Hoffer et al., [Train Longer, Generalize Better: Closing the Generalization Gap in Large Batch Training of NN](https://proceedings.neurips.cc/paper_files/paper/2017/file/a5e0ff62be0b08456fc7f1e88812af3d-Paper.pdf), Proceedings of the 31st International Conference on NEURIPS, 2017: 1729–1739.
1. Dominic Masters and Carlo Luschi, [Revisiting Small Batch Training for DNN](https://arxiv.org/pdf/1804.07612), 2018.
1. Smith, L. N. (2018). "A disciplined approach to neural network hyper-parameters: Part 1 - learning rate, batch size, momentum, and weight decay." *arXiv preprint arXiv:1803.09820*.
1. Smith, L. N., & Topin, N. (2019). *Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates*. Artificial Intelligence and Machine Learning for Multi-Domain Operations Applications. https://arxiv.org/abs/1708.07120
1. He, K.,et al (2016). "Deep Residual Learning for Image Recognition." *Proceedings of the IEEE Conference on CVPR*.
1. He, K., et al. (2019). *Bag of Tricks for Image Classification with Convolutional Neural Networks*. Proceedings of the IEEE/CVF Conference on CVPR. https://arxiv.org/abs/1812.01187
1. Goyal, P., et al. (2017). "Accurate, Large Minibatch SGD: Training ImageNet in 1 Hour." *arXiv preprint arXiv:1706.02677*.
1. Huang, G., et al (2016). *Deep Networks with Stochastic Depth*. European Conference on Computer Vision (ECCV). https://arxiv.org/abs/1603.09382
1. Vaswani, A., et al. (2017). *Attention Is All You Need*. Advances in Neural Information Processing Systems (NeurIPS). https://arxiv.org/abs/1706.03762
1. Gotmare, A. D., Keskar, N. S., Xiong, C., & Socher, R. (2018). *A Closer Look at Deep Learning Heuristics: Learning Rate Restarts, Warmup and Distillation*. International Conference on Learning Representations (ICLR). https://arxiv.org/abs/1810.12915
1. Loshchilov, I., & Hutter, F. (2017). *SGDR: Stochastic Gradient Descent with Warm Restarts*. International Conference on Learning Representations (ICLR). https://arxiv.org/abs/1608.03983
1. Loshchilov, I., & Hutter, F. (2019). *Decoupled Weight Decay Regularization*. International Conference on Learning Representations (ICLR). https://arxiv.org/abs/1711.051011.
1. Nico Van den Hooff, et al. *Online shoppers purchasing intention*. https://ubc-mds.github.io/online-shoppers-purchasing-intention/intro.html
1. Yuming Liu. *Online Shoppers Purchasing Intention*. https://rpubs.com/kingiraffe/588410

</div> </details>
